In [1]:
import numpy as np
X_train = np.load('../datasets/X_train.npy')
y_train = np.load('../datasets/y_train.npy')

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

X_train shape: (40000, 256, 37)
y_train shape: (40000,)


In [2]:
X_test = np.load('../datasets/X_test.npy')
y_test = np.load('../datasets/y_test.npy')

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_test shape: (5000, 256, 37)
y_test shape: (5000,)


In [3]:
X_val = np.load('../datasets/X_val.npy')
y_val = np.load('../datasets/y_val.npy')

print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)

X_val shape: (5000, 256, 37)
y_val shape: (5000,)


In [4]:
print("Train OCD:", sum(y_train == 1))
print("Train Non-OCD:", sum(y_train == 0))

Train OCD: 2212
Train Non-OCD: 37788


In [5]:
X_ocd = X_train[y_train == 1]
y_ocd = y_train[y_train == 1]

print("OCD samples:", X_ocd.shape)

OCD samples: (2212, 256, 37)


In [6]:
import numpy as np

def add_noise(signal):
    noise = np.random.normal(0, 0.01, signal.shape)
    return signal + noise

def time_shift(signal):
    shift = np.random.randint(1, 15)
    return np.roll(signal, shift, axis=0)

def scaling(signal):
    factor = np.random.uniform(0.95, 1.05)
    return signal * factor

In [7]:
target = 20000
augmented = []

while len(augmented) + len(X_ocd) < target:
    idx = np.random.randint(0, len(X_ocd))
    sample = X_ocd[idx]

    choice = np.random.choice(["noise", "shift", "scale"], p=[0.5, 0.3, 0.2])

    if choice == "noise":
        new_sample = add_noise(sample)
    elif choice == "shift":
        new_sample = time_shift(sample)
    else:
        new_sample = scaling(sample)

    augmented.append(new_sample)

X_aug = np.array(augmented)
y_aug = np.ones(len(X_aug))

print("Generated OCD:", X_aug.shape)

Generated OCD: (17788, 256, 37)


In [8]:
from sklearn.utils import shuffle

X_train_new = np.concatenate([X_train, X_aug], axis=0)
y_train_new = np.concatenate([y_train, y_aug], axis=0)

X_train_new, y_train_new = shuffle(X_train_new, y_train_new)

print("Final shape:", X_train_new.shape)
print("OCD count:", sum(y_train_new == 1))
print("Non-OCD count:", sum(y_train_new == 0))

Final shape: (57788, 256, 37)
OCD count: 20000
Non-OCD count: 37788


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(256, 37)))
model.add(Dropout(0.3))

model.add(LSTM(32))
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
X_train_new = X_train_new.astype('float32')
X_val = X_val.astype('float32')
X_test = X_test.astype('float32')

y_train_new = y_train_new.astype('int32')
y_val = y_val.astype('int32')
y_test = y_test.astype('int32')

In [11]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',   
    patience=5,           
    restore_best_weights=True  
)

In [13]:
import numpy as np

np.save("X_train_new.npy", X_train_new)
np.save("y_train_new.npy", y_train_new)